## Preparação do Ambiente

### Instalação de Pacotes

In [ ]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn matplotlib optuna plotly nbformat kaleido

### Importação de Bibliotecas

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import optuna
import pickle
import optuna.visualization as vis

from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

### Constantes

#### Local

In [ ]:
import sys
from pathlib import Path

# adiciona a pasta src ao sys.path
SRC_DIR = Path().resolve().parent  # .../src
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils.constants import *

#### Kaggle

In [ ]:
CATEGORICAL_COLUMNS = {
    "gestante_paciente", "raca_cor_paciente", "sigla_uf_residencia"
}

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"] 


TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

DATASET_RF_PATH = "../input/sinan-sbcas/dataset-processed-rf.csv"

## Carregamento dos Dados

In [ ]:
df = pd.read_csv(DATASET_RF_PATH)

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

## Treinamento com Validação Cruzada

In [ ]:
def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        model = RandomForestClassifier(**params)

        try:
            model.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)
            preds = model.predict(X_valid)

            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0

    return np.mean(scores), np.std(scores)


## Otimização de Hiperparâmetros

In [ ]:
FIXED_PARAMS = {
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    'max_features': 'log2'
}

### Definição da Função Objetivo

In [ ]:
def objective(trial: optuna.trial.Trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 150, 250, step=5),
        'max_depth': trial.suggest_int('max_depth', 30, 42),
        'min_samples_split': trial.suggest_int('min_samples_split', 4, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 2),
        **FIXED_PARAMS
    }
    avg, _ = train_on_folds(params)
    
    return avg

### Otimização com Optuna

In [ ]:
study = optuna.create_study(direction='maximize')
study.enqueue_trial({'n_estimators': 175, 'max_depth': 33, 'min_samples_split': 6, 'min_samples_leaf': 2})
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

In [ ]:
output_dir = Path('results/optimization/random_forest/multi')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

### Visualizações

In [ ]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [ ]:
vis.plot_param_importances(study).write_image(output_dir / 'opt_param_importances.png')
vis.plot_optimization_history(study).write_image(output_dir / 'opt_hist.png')

### Melhor Modelo

In [ ]:
BEST_PARAMS = {
    'n_estimators': 250, 
    'max_depth': 39, 
    'min_samples_split': 5, 
    'min_samples_leaf': 2
}

In [14]:
final_params = {**BEST_PARAMS, **FIXED_PARAMS}

### Média e Desvio Padrão na Validação Cruzada

In [ ]:
avg_f1_final, std_f1_final = train_on_folds(final_params)

In [ ]:
print(avg_f1_final)
print(std_f1_final)

## Avaliação Final

### Treinamento e Predição

In [ ]:
if 'early_stopping_rounds' in FIXED_PARAMS:
    del FIXED_PARAMS['early_stopping_rounds']

if 'n_iter_no_change' in FIXED_PARAMS:
    del FIXED_PARAMS['n_iter_no_change']

In [ ]:
final_model = RandomForestClassifier(**final_params)

final_model.fit(
    X_train_full, 
    y_train_full
)

y_pred_test = final_model.predict(X_test)

### Resultados

In [ ]:
print("Relatório de Classificação (Conjunto de Teste):")
print(classification_report(y_test, y_pred_test, target_names=TARGET_NAMES))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred_test, 
    display_labels=TARGET_NAMES,
    cmap='Blues',
    normalize='true',
    ax=ax
)
ax.set_title("Matriz de Confusão - Random Forest (Multiclasse)")
plt.show()